# 3.4 — Grouping and summary statistics

Compress detail into decision-sized tables while making grain, counts, denominators, missingness, and distribution explicit.

## Introduction

Use this Notebook to verify the lesson concepts with actual data and code.

## Learning outcomes

- Define the grain represented by one detail row and one summary row.
- Use groupby and named aggregation for counts, totals, statistics, and conditional counts.
- Explain the numerator and denominator of a rate, the total of a proportion, and the comparison order of a ranking.
- Validate a summary against detail and again after CSV saving and re-reading.

> **Learning route:** Required: 3.4.1–3.4.5 / Integrated practice: 3.4.6


## 3.4.1 Define the grain of detail and summary rows

One source row is a centre-month-course record. A district summary has one result row per district; a district-course summary has one per pair. Define this grain before writing `groupby()` or a correct number may still be impossible to explain.

In [ ]:
from pathlib import Path
import pandas as pd

def find_course_data(filename):
    roots = [Path.cwd(), *Path.cwd().parents, Path.home() / "work", Path("/opt/python-lab/course-materials")]
    checked = []
    for root in roots:
        for candidate in (root / "data" / filename, root / filename):
            if candidate in checked:
                continue
            checked.append(candidate)
            if candidate.is_file():
                return candidate
    raise FileNotFoundError("Course data was not found:\n" + "\n".join(map(str, checked)))

raw = pd.read_csv(find_course_data("learning-centres-practice.csv"))
clean = raw.copy()
clean["district"] = clean["district"].astype("string").str.strip().str.title()
business_key = ["centre_id", "month", "course"]
quality_problem = (
    clean["attended"].isna()
    | (clean["completed"] > clean["attended"])
    | clean.duplicated(subset=business_key, keep=False)
)
analysis = clean.loc[~quality_problem].copy()
analysis["completion_rate"] = analysis["completed"] / analysis["registered"] * 100
print("Source:", len(raw), "analysis-ready:", len(analysis), "flagged:", int(quality_problem.sum()))


## 3.4.2 Group, count, and aggregate

`groupby(key)` splits rows by matching keys, applies the same aggregations, and combines results. Named aggregation preserves output meaning. `reset_index()` returns group keys to ordinary columns for later tables and charts.

In [ ]:
district_summary = (analysis.groupby("district", dropna=False).agg(
    centre_months=("centre_id", "size"), centres=("centre_id", "nunique"),
    registered_total=("registered", "sum"), completed_total=("completed", "sum"),
).reset_index())
district_summary


### Distinguish what size, count, and nunique count

`size` counts rows including missing values, `count` counts non-missing values in one column, and `nunique` counts distinct values. Centre-month records, reported attendance values, and distinct centres are different measures.

In [ ]:
analysis.groupby("course").agg(
    rows=("centre_id", "size"), reported_attendance=("attended", "count"), distinct_centres=("centre_id", "nunique")
)


### Put a conditional count into named aggregation

“Records below 75% completion” is not an ordinary row count. First make the row-level condition a Boolean column, then sum True values within each group. Separating the decision from the aggregation lets you inspect both the rule and its count.


In [ ]:
operational = analysis.assign(low_completion=analysis["completion_rate"] < 75)
condition_summary = operational.groupby("course", as_index=False).agg(
    records=("centre_id", "size"),
    low_completion_records=("low_completion", "sum"),
)
condition_summary


## 3.4.3 Calculate totals, statistics, and rates

`sum` measures volume; `mean` the arithmetic average; `median` the middle ordered value; `min` and `max` the endpoints; and `std` spread around the mean. Extremes can pull the mean, so interpret it with count, median, and range. Standard deviation retains the variable's unit but does not explain causes.

In [ ]:
analysis.groupby("course")["registered"].agg(
    ["size", "sum", "mean", "median", "min", "max", "std"]
).round(2)


### Calculate rates from compatible aggregate totals

Overall course completion is total completed divided by total registered. Averaging row rates gives every centre-month equal weight, so it does not generally equal the rate for all learners. One answers about a typical record; the other about all registered learners.

In [ ]:
course_summary = analysis.groupby("course").agg(
    records=("centre_id", "size"), registered_total=("registered", "sum"), completed_total=("completed", "sum"),
    mean_row_completion_rate=("completion_rate", "mean"), median_row_completion_rate=("completion_rate", "median"),
    material_cost_total=("material_cost", "sum"),
)
course_summary["overall_completion_rate"] = course_summary["completed_total"] / course_summary["registered_total"] * 100
course_summary["cost_per_completion"] = course_summary["material_cost_total"] / course_summary["completed_total"]
course_summary.round(2)


### Preserve comparison hierarchy with multiple keys

District and district-course summaries have different grains. Group by both keys and sort explicitly. Never add or average results from incompatible grains without defining the intended relationship.

In [ ]:
district_course = analysis.groupby(["district", "course"], dropna=False).agg(
    records=("centre_id", "size"), registered=("registered", "sum"), completed=("completed", "sum")
).reset_index()
district_course["completion_rate"] = district_course["completed"] / district_course["registered"] * 100
district_course.sort_values(["district", "course"]).round(2)


## 3.4.4 Build indicators used for a decision

A priority order usually needs more than one rule. Here, rank higher cost per completion first, then more records, then course name. The final stable identifier makes ties reproducible. When values will be rounded for display, rank with the unrounded values.


In [ ]:
ranked_course = (
    course_summary.reset_index()
    .sort_values(
        ["cost_per_completion", "records", "course"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)
ranked_course.insert(0, "priority", range(1, len(ranked_course) + 1))
ranked_course[["priority", "course", "cost_per_completion", "records"]].round(2)


### Check the proportion denominator and 100% total

For course share within a district, divide each district-course registration count by that district's total. Dividing by the grand total answers another question. `transform('sum')` aligns each group total back to its rows so shares can be checked.

In [ ]:
district_course["district_registered_total"] = district_course.groupby("district")["registered"].transform("sum")
district_course["share_within_district"] = district_course["registered"] / district_course["district_registered_total"] * 100
print(district_course.groupby("district")["share_within_district"].sum().round(6))


## 3.4.5 Reconcile, save, and re-read the result

Summing grouped totals should reproduce the analysis-detail total. Reconcile row count, registered, completed, and material cost. Attach counts to group averages, and do not treat a difference from a small group as proof of cause or superiority.

In [ ]:
assert int(course_summary["records"].sum()) == len(analysis)
assert course_summary["registered_total"].sum() == analysis["registered"].sum()
assert course_summary["completed_total"].sum() == analysis["completed"].sum()
assert abs(course_summary["material_cost_total"].sum() - analysis["material_cost"].sum()) < 1e-9
print("Reconciliation passed")


### Save purpose-specific CSVs and validate them after re-reading

A submitted CSV is a different boundary from an in-memory DataFrame. Save two purpose-specific files, re-read them, and reconcile columns, row counts, and the first priority. Correct variables are not enough when the saved schema or order is wrong.


In [ ]:
review_columns = ["month", "centre_id", "course", "registered", "completed", "completion_rate"]
review_output = (
    operational.loc[operational["low_completion"], review_columns]
    .sort_values(["month", "centre_id", "course"])
    .reset_index(drop=True)
)
summary_output = ranked_course.copy()

output_dir = Path.cwd() / "output" / "lesson34"
output_dir.mkdir(parents=True, exist_ok=True)
review_path = output_dir / "records_to_review.csv"
summary_path = output_dir / "course_priority_summary.csv"
review_output.to_csv(review_path, index=False)
summary_output.to_csv(summary_path, index=False)

saved_review = pd.read_csv(review_path)
saved_summary = pd.read_csv(summary_path)
assert list(saved_review.columns) == review_columns
assert len(saved_review) == len(review_output)
assert list(saved_summary.columns) == list(summary_output.columns)
assert len(saved_summary) == len(summary_output)
assert saved_summary.iloc[0]["course"] == summary_output.iloc[0]["course"]
print("Saved-output reconciliation passed:", review_path, summary_path)


## 3.4.6 Integrated practice: build and validate another summary

By month and course, calculate record count, distinct centres, total registered, attended and completed, overall completion rate, material cost, and cost per completion. State every rate denominator, compare with the mean row rate, reconcile grand totals, and identify one comparison based on a small count.

In [ ]:
# Write the transfer solution here.


## Summary

- Defined result grain before using groupby.
- Connected counts, statistics, rates, proportions, and rankings to their definitions.
- Reconciled totals and verified the saved products after re-reading them.

## Next

Project 3.5A combines source inspection, quality rules, aggregation, and ranking in one operational decision using two submitted programs.

**Estimated learning time:** about 3 hours
